# LLM GMC — Name That LLM baseline

This competition-linked notebook computes a reproducible local validation score. It does not submit predictions to the leaderboard.

Baseline: word + character TF-IDF features with class-balanced LinearSVC. If `prompt_id` is present, validation is grouped by prompt to reduce prompt leakage.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
print('Dataset directory exists:', DATA_DIR.exists())
print('Dataset files:', sorted(p.name for p in DATA_DIR.glob('*')) if DATA_DIR.exists() else [])
assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found under {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
print('Train shape:', train.shape)
print('Columns:', train.columns.tolist())
assert {'text', 'model'}.issubset(train.columns), 'Expected text and model columns'
train['text'] = train['text'].fillna('').astype(str)
train['model'] = train['model'].astype(str)
print('Model classes:', train['model'].nunique())
display(train['model'].value_counts())

In [ ]:
if 'prompt_id' in train.columns and train['prompt_id'].notna().any():
    groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, valid_idx = next(splitter.split(train, groups=groups))
    split_method = 'GroupShuffleSplit by prompt_id'
else:
    train_idx, valid_idx = train_test_split(range(len(train)), test_size=0.20, random_state=42, stratify=train['model'])
    split_method = 'stratified random split (prompt_id unavailable)'

train_part = train.iloc[list(train_idx)].copy()
valid_part = train.iloc[list(valid_idx)].copy()
print('Validation method:', split_method)
print('Training rows:', len(train_part))
print('Validation rows:', len(valid_part))
print('Validation classes:', valid_part['model'].nunique())

In [ ]:
features = FeatureUnion([
    ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=180000)),
    ('char', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=2, sublinear_tf=True, max_features=220000)),
])
model = Pipeline([('features', features), ('classifier', LinearSVC(C=2.0, class_weight='balanced'))])

started = time.time()
print('Training baseline...')
model.fit(train_part['text'], train_part['model'])
train_seconds = time.time() - started
print(f'Training completed in {train_seconds:.1f} seconds')
pred = model.predict(valid_part['text'])
accuracy = float(accuracy_score(valid_part['model'], pred))
balanced_accuracy = float(balanced_accuracy_score(valid_part['model'], pred))
print('\n' + '=' * 48)
print(f'BASELINE ACCURACY:          {accuracy:.4f} ({accuracy * 100:.2f}%)')
print(f'BASELINE BALANCED ACCURACY: {balanced_accuracy:.4f} ({balanced_accuracy * 100:.2f}%)')
print('=' * 48)
print(classification_report(valid_part['model'], pred, zero_division=0))

results = {'experiment':'LLM GMC TF-IDF + LinearSVC baseline','competition':'name-that-llm','split_method':split_method,'train_rows':int(len(train_part)),'validation_rows':int(len(valid_part)),'total_rows':int(len(train)),'model_classes':int(train['model'].nunique()),'accuracy':accuracy,'accuracy_percent':round(accuracy*100,4),'balanced_accuracy':balanced_accuracy,'balanced_accuracy_percent':round(balanced_accuracy*100,4),'training_seconds':round(train_seconds,2)}
out = Path('/kaggle/working/validation_results.json')
out.write_text(json.dumps(results, indent=2))
print('Saved metrics:', out)
print(json.dumps(results, indent=2))

## Result interpretation

The accuracy printed above is a local validation score, not the Kaggle public leaderboard score. No leaderboard submission is made in this notebook. Review this score before choosing the next experiment.